In [1]:
# The previous notebook was Example_Data_Creation where we created .pkl's to save our data

In [79]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [80]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
import time
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [81]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [82]:
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/09 11:24:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/09 11:24:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [83]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context


In [84]:
%%time
# This is how we would load the .pkl's back in (it can take a minute so be patient)
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_extra_features.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_extra_features.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

CPU times: user 23.7 s, sys: 321 ms, total: 24 s
Wall time: 24.2 s


In [85]:
type(group_a_spark_df_loaded)

pyspark.sql.dataframe.DataFrame

In [86]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df_loaded
cntrl_df = group_c_spark_df_loaded

In [87]:
print(alz_df.count())
print(cntrl_df.count())
print(alz_df.select("SubjectID", "EpochID").count())
print(cntrl_df.select("SubjectID", "EpochID").count())
print(alz_df.select("SubjectID", "EpochID").distinct().count())
print(cntrl_df.select("SubjectID", "EpochID").distinct().count())

25/04/09 11:25:07 WARN TaskSetManager: Stage 0 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.
25/04/09 11:25:08 WARN TaskSetManager: Stage 3 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


1032175
858040


25/04/09 11:25:08 WARN TaskSetManager: Stage 6 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.


1032175


25/04/09 11:25:09 WARN TaskSetManager: Stage 9 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


858040


25/04/09 11:25:09 WARN TaskSetManager: Stage 12 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.


10865


25/04/09 11:25:09 WARN TaskSetManager: Stage 18 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


9032


In [88]:
# Now lets do dimensionality reducton by first normalizing the power and then doing PCA.
# First step is lets split the data into training/testing

In [89]:
from dimensionality_reduction import normalize_power
# *REFERENCE* df_a_norm, df_c_norm = normalize_power(result_group_a, result_group_c)

In [90]:
cntrl_df.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [91]:
NUM_TEST_SUBJECTS_PER_GROUP = 2  # i know before we had three , but 2 is better 3 took out too much data.

alz_test_subjects = (
    alz_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


cntrl_test_subjects = (
    cntrl_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


25/04/09 11:25:10 WARN TaskSetManager: Stage 24 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.
25/04/09 11:25:10 WARN TaskSetManager: Stage 28 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


In [92]:
print(f"azl test subjects {alz_test_subjects}\ncntrl test subjects {cntrl_test_subjects}")

azl test subjects ['sub-001', 'sub-002']
cntrl test subjects ['sub-037', 'sub-038']


In [93]:
# now we put the labels on both datasets , and split it up into testing/training

In [94]:
from pyspark.sql.functions import lit

In [95]:
# making alz have the label 1 and cntrl 0 for the ml portion ahea

In [96]:
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [97]:
# Filter test rows
alz_test_df = alz_df.filter(alz_df.SubjectID.isin(alz_test_subjects))
cntrl_test_df = cntrl_df.filter(cntrl_df.SubjectID.isin(cntrl_test_subjects))

# Filter training rows (not in test subjects)
alz_train_df = alz_df.filter(~alz_df.SubjectID.isin(alz_test_subjects))
cntrl_train_df = cntrl_df.filter(~cntrl_df.SubjectID.isin(cntrl_test_subjects))


In [98]:
# Individual counts
alz_train_count = alz_train_df.select("SubjectID", "EpochID").distinct().count()
alz_test_count = alz_test_df.select("SubjectID", "EpochID").distinct().count()
cntrl_train_count = cntrl_train_df.select("SubjectID", "EpochID").distinct().count()
cntrl_test_count = cntrl_test_df.select("SubjectID", "EpochID").distinct().count()

# Combined counts
test_total_count = alz_test_df.unionByName(cntrl_test_df).select("SubjectID", "EpochID").distinct().count()
train_total_count = alz_train_df.unionByName(cntrl_train_df).select("SubjectID", "EpochID").distinct().count()

# Print them out
print(f"Alzheimer's train: {alz_train_count}")
print(f"Alzheimer's test:  {alz_test_count}")
print(f"Control train:     {cntrl_train_count}")
print(f"Control test:      {cntrl_test_count}")
print(f"Total test:        {test_total_count}")
print(f"Total train:       {train_total_count}")

25/04/09 11:25:10 WARN TaskSetManager: Stage 32 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.
25/04/09 11:25:11 WARN TaskSetManager: Stage 53 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


Alzheimer's train: 10350
Alzheimer's test:  515
Control train:     8414
Control test:      618
Total test:        1133
Total train:       18764


In [99]:
train_df = alz_train_df.unionByName(cntrl_train_df)
test_df = alz_test_df.unionByName(cntrl_test_df)


In [100]:
from dimensionality_reduction import normalize_power # it z-scores the data
# NOTE, this uses the first parameters for mean and std for the z-score, so none of test_df's data is used to z-score
train_df, test_df = normalize_power(train_df, test_df) 

In [101]:
from dimensionality_reduction import prepare_features_for_pca
# this pivots the tables so that its better suited for PCA and ML with pyspark's libraries
print(train_df.columns)
train_df, train_features_column = prepare_features_for_pca(train_df)
test_df, test_features_column = prepare_features_for_pca(test_df)
print(train_df.columns) # as we can see after they get flatened 

['Electrode', 'WaveBand', 'SubjectID', 'EpochID', 'Power', 'label']
['SubjectID', 'EpochID', 'label', 'Fz_Delta', 'F3_Theta', 'T4_Theta', 'Pz_Beta', 'T3_Alpha', 'Fp2_Total', 'F3_Beta', 'T5_Delta', 'F3_Total', 'T6_Theta', 'Pz_Theta', 'F7_Alpha', 'F3_Delta', 'T3_Beta', 'T4_Beta', 'Pz_Delta', 'F4_Delta', 'P4_Alpha', 'Fp2_Alpha', 'T6_Beta', 'C4_Theta', 'P4_Total', 'P3_Total', 'Fp1_Theta', 'T5_Total', 'O1_Delta', 'F3_Alpha', 'C3_Total', 'T4_Total', 'O1_Theta', 'C4_Beta', 'Fz_Alpha', 'C3_Beta', 'O1_Alpha', 'Fz_Theta', 'F7_Total', 'P4_Beta', 'Fz_Beta', 'Cz_Total', 'F7_Theta', 'Fp2_Beta', 'T4_Delta', 'T3_Total', 'F8_Beta', 'Pz_Total', 'O2_Theta', 'P4_Theta', 'Cz_Beta', 'T4_Alpha', 'Cz_Delta', 'F8_Total', 'F4_Total', 'C4_Total', 'F8_Delta', 'P3_Alpha', 'O1_Beta', 'Fp2_Theta', 'Cz_Alpha', 'O2_Delta', 'Fp2_Delta', 'P3_Delta', 'Fp1_Total', 'Fp1_Beta', 'Pz_Alpha', 'O2_Alpha', 'P3_Theta', 'O1_Total', 'F8_Alpha', 'C3_Theta', 'T5_Beta', 'T3_Delta', 'F7_Delta', 'O2_Beta', 'F4_Alpha', 'T6_Total', 'Fp1_A

In [102]:
if train_features_column != test_features_column:
    print("!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!")

!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!


In [103]:
from dimensionality_reduction import fit_pca_model
#Note , this finds the features to explain the model's PCA
K_VAR_TARGET=0.95
pca_model_func, k_val = fit_pca_model(train_df, train_features_column, variance_target=K_VAR_TARGET)

In [104]:
print(f"We can explian {K_VAR_TARGET} with {k_val} features. That is a lot less then {len(train_df.columns)-3} (we hope).") #-3 for subjectID , epochID and lebel

We can explian 0.95 with 18 features. That is a lot less then 95 (we hope).


In [105]:
# know that we know we can explain 95% of the variance (or what we set target to) ,
# lets make our dataframces only have those important columns
from dimensionality_reduction import apply_pca_model
train_df = apply_pca_model(train_df, train_features_column, pca_model_func, k_val)
test_df  = apply_pca_model(test_df, train_features_column, pca_model_func, k_val)

In [106]:

train_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = false)



In [107]:
# Now that our data is labled and reduced in size, now we can do ML 

In [108]:
%%time
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Get input size from PCA features

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=[k_val, 100, 2],  # input → hidden (100 units) → 2 output classes
    maxIter=1000,
    seed=42
)

mlp_model = mlp.fit(train_df)
mlp_preds = mlp_model.transform(test_df)

CPU times: user 9.1 ms, sys: 9.16 ms, total: 18.3 ms
Wall time: 55 s


In [109]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mlp_auc = evaluator.evaluate(mlp_preds)

print("MLP AUC:", mlp_auc)

MLP AUC: 0.6535237377069784


In [110]:
preds_pd = mlp_preds.select("prediction", "label").toPandas()

from sklearn.metrics import classification_report, accuracy_score

print("Neural Network accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

Neural Network accuracy: 0.6301853486319505
              precision    recall  f1-score   support

     Control       0.68      0.60      0.64       618
 Alzheimer's       0.58      0.67      0.62       515

    accuracy                           0.63      1133
   macro avg       0.63      0.63      0.63      1133
weighted avg       0.64      0.63      0.63      1133



In [111]:
 # hmm , less support then expected, what is heppening , something cutting it off, need ot check size of train

# MORE ML models

In [112]:
from pyspark.ml.classification import LinearSVC

# Train SVM model
svm = LinearSVC(featuresCol="features", labelCol="label", maxIter=100, regParam=0.1)
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)

# Evaluate SVM
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
svm_auc = evaluator.evaluate(svm_preds)
print("SVM AUC:", svm_auc)

# Accuracy and report
preds_pd = svm_preds.select("prediction", "label").toPandas()
from sklearn.metrics import classification_report, accuracy_score

print("SVM accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


SVM AUC: 0.694931975995224
SVM accuracy: 0.5454545454545454
              precision    recall  f1-score   support

     Control       0.70      0.29      0.41       618
 Alzheimer's       0.50      0.85      0.63       515

    accuracy                           0.55      1133
   macro avg       0.60      0.57      0.52      1133
weighted avg       0.61      0.55      0.51      1133



In [113]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
tree_model = tree.fit(train_df)
tree_preds = tree_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
tree_auc = evaluator.evaluate(tree_preds)
print("Decision Tree AUC:", tree_auc)

preds_pd = tree_preds.select("prediction", "label").toPandas()
print("Decision Tree accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


Decision Tree AUC: 0.6871005749835044
Decision Tree accuracy: 0.5851721094439541
              precision    recall  f1-score   support

     Control       0.69      0.43      0.53       618
 Alzheimer's       0.53      0.77      0.63       515

    accuracy                           0.59      1133
   macro avg       0.61      0.60      0.58      1133
weighted avg       0.62      0.59      0.57      1133



In [114]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=100)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
gbt_auc = evaluator.evaluate(gbt_preds)
print("Gradient Boosted Trees AUC:", gbt_auc)

preds_pd = gbt_preds.select("prediction", "label").toPandas()
print("GBT accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/admin/neuro-venv/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/admin/neuro-venv/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socket.py", line 704, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col

# Step 1: Fit LSH model on training data
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=1.0,
    numHashTables=3
)
lsh_model = lsh.fit(train_df)

# Step 2: Perform approximate similarity join between test and train
# This will find the approximate nearest neighbors of test samples in train set
similarities = lsh_model.approxSimilarityJoin(
    datasetA=test_df,
    datasetB=train_df,
    threshold=float("inf"),  # You can limit this if needed
    distCol="euclidean_distance"
)

# Step 3: For each test point, pick nearest neighbor (smallest distance)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("datasetA").orderBy("euclidean_distance")

nearest_neighbors = similarities \
    .withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1)

# Step 4: Collect prediction from nearest training label
predictions = nearest_neighbors.select(
    col("datasetA.label").alias("true_label"),
    col("datasetB.label").alias("predicted_label")
)

# Step 5: Evaluate
preds_pd = predictions.toPandas()

from sklearn.metrics import accuracy_score, classification_report

print("Approximate KNN accuracy:", accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))


In [ ]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col

# Step 1: Fit LSH model on training data
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=0.25,
    numHashTables=6
)
lsh_model = lsh.fit(train_df)

# Step 2: Perform approximate similarity join between test and train
# This will find the approximate nearest neighbors of test samples in train set
similarities = lsh_model.approxSimilarityJoin(
    datasetA=test_df,
    datasetB=train_df,
    threshold=float("inf"),  # You can limit this if needed
    distCol="euclidean_distance"
)

# Step 3: For each test point, pick nearest neighbor (smallest distance)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("datasetA").orderBy("euclidean_distance")

nearest_neighbors = similarities \
    .withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1)

# Step 4: Collect prediction from nearest training label
predictions = nearest_neighbors.select(
    col("datasetA.label").alias("true_label"),
    col("datasetB.label").alias("predicted_label")
)

# Step 5: Evaluate
preds_pd = predictions.toPandas()

from sklearn.metrics import accuracy_score, classification_report

print("Approximate KNN accuracy:", accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))


25/04/09 11:27:37 WARN DAGScheduler: Broadcasting large task binary with size 1000.3 KiB
25/04/09 11:27:37 WARN DAGScheduler: Broadcasting large task binary with size 1000.8 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1001.5 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1002.5 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1007.4 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1007.9 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1008.6 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1009.6 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1011.6 KiB
25/04/09 11:27:38 WARN DAGScheduler: Broadcasting large task binary with size 1013.9 KiB
25/04/09 11:27:38 WAR

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Convert Spark DataFrame to pandas
preds_pd = predictions.toPandas()

# Accuracy
acc = accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"])
print("Approximate KNN accuracy:", acc)

# Classification report
target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))

# Optional: Confusion matrix
cm = confusion_matrix(preds_pd["true_label"], preds_pd["predicted_label"])
sns.heatmap(cm, annot=True, fmt="d", xticklabels=target_names, yticklabels=target_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Approximate KNN (LSH)")
plt.show()
